# 3.3 Object Detection with LLMs (Doubao Version)

## Object Detection
In the previous lesson, we used a multimodal LLM for image understanding. If we need to approach a specific target, we need to know the target's position -- this is what object detection provides: bounding boxes (bbox).

<img src="img/bbox.png" width='640px' />


### **Object Detection**
**Object Detection** is a computer vision task that identifies objects in images or video and locates their positions (typically represented by bounding boxes). It not only classifies objects ("person", "car", "dog") but also localizes them within the image. Unlike **image classification** (single label) and **image segmentation** (pixel-level), object detection handles multiple objects of multiple classes simultaneously.

---

#### 1. **Traditional Methods** (pre-2010)
- **Pipeline**: Sliding window generation -> Feature extraction (SIFT, HOG) -> Classifier (SVM) classification.
- **Disadvantages**: High computational cost, poor robustness, limited to specific scenes.

#### 2. **Deep Learning Methods** (2012-2020)
- **Pipeline**:
  1. **Feature extraction**: Using CNNs to extract image features
  2. **Bounding box regression**: Predicting positions (anchor-based or anchor-free)
  3. **Classification and filtering**: Using IoU for evaluation, Non-Maximum Suppression (NMS) for filtering

- **Categories**:
  1. **Two-Stage** (region-based): Generate proposals first (Faster R-CNN with RPN), then classify. Higher precision but slower.
  2. **One-Stage** (direct): Predict boxes and classes simultaneously (YOLO). Faster but may struggle with small objects.
  3. **YOLO series**: Real-time, fast, widely used in monitoring scenarios.


#### 3. **Current State-of-the-Art**

<img src="img/s3-1.png" width='640px' />

1. Attention-based, Transformer-Based (DETR): No NMS needed, but training is resource-intensive.
2. Under the LLM paradigm, moving from closed-set (predefined classes) to open-set (open vocabulary) detection.
3. Able to detect targets based on natural language descriptions or examples, enabling zero-shot detection with strong multimodal alignment and generalization.


Open-vocabulary object detection with LLMs:

<img src="img/dino.jpg" width='800px' />

## Resources

- DINO open-source: https://github.com/IDEA-Research/DINO
- Grounding-DINO open-source: https://github.com/IDEA-Research/GroundingDINO
- DINO online playground: https://cloud.deepdataspace.com/playground/grounding_dino
- DINO API docs: https://cloud.deepdataspace.com/docs#model/dinox


<img src='img/dino-s.png' width='720px' />

Grounding DINO: 0.1 yuan/call

Registration includes 20 free credits, approximately 200 calls.

Get your token: https://cloud.deepdataspace.com/dashboard/token-key

In [ ]:
!pip install openai

In [ ]:
!pip install opencv-python

In [ ]:
# openai and opencv-python should already be installed, no additional packages needed

In [ ]:
# Note: After installation completes, you may need to restart the kernel

In [ ]:
import base64
import json
import re
import cv2
from openai import OpenAI

ARK_API_KEY = "YOUR_IO_API_KEY" # API Key
MODEL = "moonshotai/Kimi-K2.6"

client = OpenAI(
base_url="https://api.intelligence.io.solutions/api/v1",
api_key="YOUR_IO_API_KEY",
)

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def detect_objects(image_path, text_prompt, output_dir="./img"):
    image = cv2.imread(image_path)
    h, w = image.shape[:2]
    b64 = encode_image(image_path)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": (
                    f"Detect the following objects in the image: {text_prompt}. The image is {w}x{h} pixels. "
                    "Return results in JSON format: {\"objects\": [{\"bbox\": [x1, y1, x2, y2], "
                    "\"category\": \"class_name\", \"score\": 0.9}]}. bbox values are in pixels. Return only JSON."
                )}
            ]
        }],
    )
    content = response.choices[0].message.content
    match = re.search(r'\{.*\}', content, re.DOTALL)
    result = json.loads(match.group()) if match else {"objects": []}

    for obj in result.get("objects", []):
        x1, y1, x2, y2 = [int(v) for v in obj["bbox"]]
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"{obj['category']}: {obj.get('score', 0):.2f}"
        cv2.putText(image, label, (x1, max(y1 - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imwrite(output_dir + "/annotated_image.jpg", image)
    return result

img_path = "./img/p-4-1-s.png"
output_path = "./img"
text = "yellow duck.Cola"
result = detect_objects(img_path, text, output_path)

In [ ]:
from IPython.display import display # type: ignore
from PIL import Image
img_path = "./img/annotated_image.jpg"
image_pil = Image.open(img_path)
display(image_pil)

## Reading Images with OpenCV

In [ ]:
import cv2 # type: ignore
local_image_path = "./img/p-4-1-s.png"
image = cv2.imread(local_image_path)
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
import uuid
bgr_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
# Generate random filename (with extension)
file_name = f"random_{uuid.uuid4().hex}.png"  # Example: random_1a2b3c4d5e.png
cv2.imwrite('./img/' + file_name, bgr_image)

In [ ]:
results = detect_objects("./img/" + file_name, "yellow duck.Cola", "./img")

In [ ]:
results